# BUCSS 2026
## AI Session Hands-On Part 2: Training different models

In this notebook you will:
- Load the training data
- Train AI models
- Test on different data sets

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier


#These are internal
from scripts.splitting_and_training import split_data, train_model



First we need to decide if we want every column in our dataframe as a feature in our ML model. 

In [ ]:
df = pd.read_parquet("../dataframes_for_training/dortmund_training.parquet")
df.columns

In [4]:
df_training = df.drop(columns=["air_temperature", 
                        "lcz_nearest", "datetime_utc","u10", 
                        "v10", "ssrd", "tp", "d2m", "t2m"]).copy()

df["temp_diff"] = df["air_temperature"] - df["t2m_corr"]

target_var = df["temp_diff"]  #df["air_temperature"]


In [5]:
X = df_training
y = target_var
mode = "block_spatial" # "random", "random_spatial", "block_spatial"

Data split should be model_type agnostic !

In [6]:
X_train, X_val, y_train, y_val = split_data(X, y, mode) 

## XGBoost

In [ ]:
y_pred, model = train_model(X_train, X_val, y_train, y_val, model_type = "xgboost")

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

t2m_corr_test = df.loc[y_val.index, "t2m_corr"]

y_rec = t2m_corr_test.values + y_val.values          # observed temp reconstructed
y_pred_rec = t2m_corr_test.values + y_pred   

rmse = np.sqrt(mean_squared_error(y_rec, y_pred_rec))
r2   = r2_score(y_rec, y_pred_rec)
print(f"RMSE: {rmse:.3f} °C   R²: {r2:.3f}")



In [ ]:
rmse_era5 = np.sqrt(mean_squared_error(t2m_corr_test, y_rec))
r2_era5   = r2_score(y_rec, t2m_corr_test)
print(f"ERA5 RMSE: {rmse_era5:.3f} °C   R²: {r2_era5:.3f}")


In [ ]:
import cmcrameri.cm as cmc

def importance(model):
    # For models trained with xgb.train() and DMatrix with feature_names
    feature_names = model.feature_names
    cmap = cmc.batlow
    # sample two colors away from extremes for clarity
    predicted_color = cmap(0.7)
        # Get importance scores (you can choose different types)
    importance_dict = model.get_score(importance_type='total_gain')  # or 'gain', 'cover'
    
    # Convert to arrays, ensuring order matches feature_names
    importances = np.array([importance_dict.get(fname, 0.0) for fname in feature_names])

    sorted_indices = np.argsort(importances)
    sorted_importances = importances[sorted_indices]
    sorted_features = [feature_names[i] for i in sorted_indices]

    # Create the horizontal bar plot
    plt.figure(figsize=(12, max(8, len(sorted_features) * 0.3)))
    bars = plt.barh(range(len(sorted_features)), sorted_importances, 
                    color=predicted_color,
                    #  alpha=0.7
                    )

    # Customize the plot
    plt.yticks(range(len(sorted_features)), sorted_features)
    plt.xlabel('Total gain')
    plt.ylabel('Features')
    # plt.title('XGBoost Feature Importances')
    plt.grid(axis='x', alpha=0.3)

        # # Add value labels on bars
        # for i, (bar, v) in enumerate(zip(bars, sorted_importances)):
        #     plt.text(v + max(sorted_importances) * 0.01, i, f'{v:.3f}', 
        #             va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    print("\nFeature importances (sorted):")
    for feat, val in zip(sorted_features[::-1], sorted_importances[::-1]):
        print(f"{feat}: {val:.6f}")
    
importance(model)

## Land Use Regression

Which variables should be kept ? Once we've figured that out, we should also determine what target variable we want to train for...

In [8]:
df_training = df.drop(columns=["air_temperature"]).copy()
target_var = df["air_temperature"]

import numpy as np

df["temp_diff"] = df["air_temperature"] - df["t2m_corr"]

# cyclical time features
hod = df["datetime_utc"].dt.hour                    # 0-23
doy = df["datetime_utc"].dt.dayofyear               # 1-365/366

df["tod_sin"] = np.sin(2 * np.pi * hod / 24)
df["tod_cos"] = np.cos(2 * np.pi * hod / 24)
df["doy_sin"] = np.sin(2 * np.pi * doy / 365)
df["doy_cos"] = np.cos(2 * np.pi * doy / 365)


In [9]:
import statsmodels.api as sm
import numpy as np
from sklearn.metrics import r2_score

def val_r2(X_tr, y_tr, X_val, y_val, cols):
    """Fit OLS on train cols, return R² on validation. 0 if no cols."""
    if not cols:
        return 0.0
    model = sm.OLS(y_tr, sm.add_constant(X_tr[cols])).fit()
    pred = model.predict(sm.add_constant(X_val[cols], has_constant="add"))
    return r2_score(y_val, pred)


def select_predictors(X_tr, y_tr, X_val, y_val, geo_candidates):
    selected = []
    score = lambda cols: val_r2(X_tr, y_tr, X_val, y_val, cols)

    def try_group(options):
        """options: list of column-lists (each an alternative). Keep the best if it beats current."""
        nonlocal selected
        base = score(selected)
        best_opt, best_s = None, base
        for opt in options:
            if opt and not all(c in X_tr.columns for c in opt):
                continue
            s = score(selected + opt)
            if s > best_s:
                best_opt, best_s = opt, s
        if best_opt:
            selected += best_opt

    # met groups (pick one alternative each)
    try_group([["t2m"], ["t2m_corr"]])
    try_group([["d2m"], ["rh"]])
    try_group([["u10", "v10"], ["wspd"]])

    # lat/lon: both or neither
    try_group([["latitude", "longitude"], []])

    # geo families: best member of each (tcd, imp, bh, dtm, ...)
    fams = {}
    for c in geo_candidates:
        fams.setdefault(c.split("_")[0], []).append(c)
    for members in fams.values():
        try_group([[m] for m in members])      # each radius is a single-col alternative

    return selected, score(selected)

We train the LUR on hourly means of temp_diff, with selectable months


In [12]:
months = range(1, 13)  #[6, 7, 8]

met_cols = {"t2m", "t2m_corr", "d2m", "rh", "u10", "v10", "wspd"}
exclude = {"station_id", "datetime_utc", "air_temperature", "temp_diff", "lcz_nearest"}

geo_candidates = [c for c in df.columns
                  if c not in exclude and c not in met_cols
                  and c not in ("latitude", "longitude")      # <- add this
                  and not c.startswith(("LCZ_", "tod_", "doy_"))]

# --- build per-station-per-hour means over chosen months ---
sub = df[df["datetime_utc"].dt.month.isin(months)].copy()
sub["hour"] = sub["datetime_utc"].dt.hour

geo_and_met = geo_candidates + list(met_cols)
agg = {c: "first" for c in geo_and_met if c in sub.columns}
agg["latitude"] = "first"          # needed for the spatial split, not a predictor
agg["longitude"] = "first"
for m in met_cols:
    if m in sub.columns:
        agg[m] = "mean"
agg["temp_diff"] = "mean"

station_hour = sub.groupby(["station_id", "hour"]).agg(agg).reset_index()



In [ ]:
# build the per-station-per-hour means as before -> station_hour (must still have station_id, lat, lon)
results = {}
for hour in range(24):
    df_h = station_hour[station_hour["hour"] == hour]

    X = df_h.drop(columns=["temp_diff"])          # keep station_id/lat/lon for splitting
    y = df_h["temp_diff"]
    X_tr, X_val, y_tr, y_val = split_data(X, y, mode="block_spatial", quadrant="ne")

    sel, r2 = select_predictors(X_tr, y_tr, X_val, y_val, geo_candidates)
    results[hour] = {"predictors": sel, "val_r2": r2}
    print(f"hour {hour:02d}: val_R²={r2:.3f}  {sel}")

In [ ]:
import statsmodels.api as sm

# for one hour, using that hour's selected predictors
hour = 2
sel = results[hour]["predictors"]
df_h = station_hour[station_hour["hour"] == hour]

X = sm.add_constant(df_h[sel])
model = sm.OLS(df_h["temp_diff"], X).fit()

print(model.params)          # just the coefficients, one per predictor + const

In [ ]:
# candidates: same geo, but now INCLUDE cyclical time (they vary across the full data)
met_cols = {"t2m", "t2m_corr", "d2m", "rh", "u10", "v10", "wspd"}
exclude = {"station_id", "datetime_utc", "air_temperature", "temp_diff", "lcz_nearest"}

geo_candidates = [c for c in df.columns
                  if c not in exclude and c not in met_cols
                  and c not in ("latitude", "longitude")
                  and not c.startswith("LCZ_")
                  and not c.startswith(("tod_", "doy_"))]     # keep geo only here

# --- one model on the FULL data (no per-hour, no means) ---
X = df.drop(columns=["temp_diff"])       # keep station_id/lat/lon for the split
y = df["temp_diff"]

X_tr, X_val, y_tr, y_val = split_data(X, y, mode="block_spatial", quadrant="ne")

sel, r2 = select_predictors(X_tr, y_tr, X_val, y_val, geo_candidates)
print(f"full-data model: val_R²={r2:.3f}")
print("selected:", sel)

## Scaling ?


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaled_array = scaler.fit_transform(df[candidates])     # fit AND transform, on the candidates

scaled_array

In [ ]:
results = {}
for hour in range(24):
    sub = station_hour[station_hour["hour"] == hour]
    selected = forward_select(sub, "temp_diff", geo_predictors, threshold=0.001)
    if selected:
        X = sm.add_constant(sub[selected])
        r2 = sm.OLS(sub["temp_diff"], X).fit().rsquared_adj
    else:
        r2 = 0.0
    results[hour] = {"predictors": selected, "adj_r2": r2}
    print(f"hour {hour:02d}: adj_R²={r2:.3f}  {selected}")

In [ ]:
import statsmodels.api as sm

hour = 3
sub = station_hour[station_hour["hour"] == hour]

selected = results[hour]["predictors"]          # <- get THIS hour's selected set
X = sm.add_constant(sub[selected])
y = sub["temp_diff"]

model = sm.OLS(y, X).fit()
print(model.summary())